# 생성형 AI와 함께하는 파이썬 데이터 분석 — 실습 노트북 (Google Colab)
충북과학기술혁신원 · 1~6모듈 공통

**사용 데이터 (업체 제공)**
- `2. 라인생산_ 검사값 대 온도_월별 Raw Data.xlsx` — 월별 9개 시트, a~d 검사값과 건조온도(1~3존)
- `5. 공정데이터_MES_Tag Data.xlsx` — MES 태그(관 온도 등) 시계열
- `라인_센서로그_가상데이터.csv` — 부록(분류 심화)용 **교육용 가상 데이터**

> **시작 순서** : ① 왼쪽 📁 파일 아이콘 → 엑셀 2개와 CSV를 끌어다 놓기 ② 아래 【준비】 셀 실행 ③ 위에서부터 셀을 하나씩 `Shift + Enter`
> 오류가 나면 오류 메시지를 복사해 ChatGPT에 붙여넣고 "어떤 상황인지 + 고쳐줘"라고 요청합니다.
> 셀 제목의 번호는 교안 슬라이드의 실습 번호와 같습니다.

## 【준비】 한글 폰트 설정
그래프에 한글이 □□로 깨지면 **런타임 → 세션 다시 시작** 후 이 셀부터 다시 실행하세요.

In [ ]:
# 【준비】 한글 폰트 설치 (Colab · 처음 한 번만)
!sudo apt-get -qq -y install fonts-nanum
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
fm._load_fontmanager(try_read_cache=False)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
print('한글 폰트 설정 완료')

---
# 1모듈 · 첫 실행 【1-4】

In [ ]:
# ① 글자 보여주기
print("A라인 데이터 분석 시작")

In [ ]:
# ② 담고 → 보여주기
temperature = 65.2
print(temperature)

## 【미니①】 ★ ChatGPT로 첫 그래프 받기 — 라인별 오늘 생산량
ChatGPT에게 아래처럼 시키고, 받은 코드를 새 셀에 붙여넣어 실행합니다. 막히면 아래 셀을 실행하세요.
> Colab에서 실행할 파이썬 코드를 짜줘. A·B·C·D 라인의 오늘 생산량이 1200, 980, 1150, 870개야. 막대그래프로 그리고, 목표 1000개에 빨간 점선을 그어줘. 목표를 넘은 라인은 파란색, 못 넘은 라인은 주황색으로 칠하고 막대 위에 숫자를 써줘. 제목과 축 이름은 한글로.

In [ ]:
# 【미니①】 라인별 오늘 생산량 막대그래프
import matplotlib.pyplot as plt

라인 = ['A라인', 'B라인', 'C라인', 'D라인']
생산량 = [1200, 980, 1150, 870]
목표 = 1000

색 = ['#1971C2' if v >= 목표 else '#E8590C' for v in 생산량]   # 목표 이상 파랑, 미달 주황
plt.figure(figsize=(8, 4.5))
막대 = plt.bar(라인, 생산량, color=색)
plt.axhline(목표, color='red', linestyle='--', label='목표 1,000개')
for b, v in zip(막대, 생산량):
    plt.text(b.get_x() + b.get_width() / 2, v + 15, f'{v:,}', ha='center', fontsize=12)
plt.title('라인별 오늘 생산량')
plt.xlabel('라인')
plt.ylabel('생산량 (개)')
plt.legend()
plt.show()

## 【미니②】 ★ 파일을 읽어 그래프로 — 설비별 불량률과 평균 온도
`라인_센서로그_가상데이터.csv`를 📁 파일 패널에 올린 뒤, ChatGPT에게 아래처럼 시킵니다. 막히면 아래 셀을 실행하세요.
> Colab에 올린 '라인_센서로그_가상데이터.csv' 파일을 읽어줘. 열은 일시, 설비 ID, 온도, 압력, 생산량, 불량여부(1=불량, 0=정상)야. 설비 ID별 불량률(%)과 평균 온도를 막대그래프 2개로 나란히 그려줘. 불량률이 가장 높은 설비는 빨간색, 나머지는 회색으로 하고 막대 위에 숫자를 써줘. 제목과 축 이름은 한글로.

In [ ]:
# 【미니②】 설비별 불량률(%)과 평균 온도
import pandas as pd
import matplotlib.pyplot as plt

센서 = pd.read_csv('라인_센서로그_가상데이터.csv')
print(센서.shape)                                        # (2400, 6)

불량률 = 센서.groupby('설비 ID')['불량여부'].mean() * 100  # 불량여부 평균 = 불량률
평균온도 = 센서.groupby('설비 ID')['온도'].mean()

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for a, 값, 제목, 단위 in [(ax[0], 불량률, '설비별 불량률', '불량률 (%)'), (ax[1], 평균온도, '설비별 평균 온도', '온도 (℃)')]:
    색 = ['#C92A2A' if i == 불량률.idxmax() else '#ADB5BD' for i in 값.index]
    막대 = a.bar(값.index, 값.values, color=색)
    for b, v in zip(막대, 값.values):
        a.text(b.get_x() + b.get_width() / 2, v, f'{v:.1f}', ha='center', va='bottom', fontsize=11)
    a.set_title(제목); a.set_xlabel('설비 ID'); a.set_ylabel(단위)
plt.tight_layout()
plt.show()

---
# 3모듈 · 데이터 결측값 처리 실습 (2번 파일, 첫 시트)
엑셀 파일을 그대로 불러와 보고, 빈칸(NaN)을 확인하고, 채우고, 새 파일로 저장합니다.

In [ ]:
# 【3-①】 Excel 데이터 불러오기
import pandas as pd

# Excel 데이터를 Python으로 가져오는 명령
df = pd.read_excel("2. 라인생산_ 검사값 대 온도_월별 Raw Data.xlsx")

# 데이터의 맨 위 5개 행을 화면에 보여달라는 명령
print(df.head())

In [ ]:
# 【3-②】 데이터 구조 확인
# 데이터가 몇 행 x 몇 열인가?
print(df.shape)

# 어떤 항목들이 있는가?
print(df.columns)

In [ ]:
# 【3-③】 결측치 확인
print(df.isnull().sum())

### 【3-⑤】 날짜의 빈칸 채우기
`df['구분']`으로 바로 채우면 **KeyError** 가 납니다. 실제 열 이름이 `'구   분'`(띄어쓰기 포함)이기 때문입니다.
오류 메시지를 ChatGPT에 붙여넣으면 아래처럼 **열 이름을 바꾼 뒤 채우는 코드**를 받을 수 있습니다.

In [ ]:
# 【3-⑤】 열 이름 확인 → 첫 번째 열 이름을 '구분'으로 변경 → 빈칸 채우기
print(df.columns.tolist())

df.rename(columns={df.columns[0]: '구분'}, inplace=True)
print(df.columns.tolist())

# ffill : forward fill, 즉 앞의 값으로 빈칸을 채운다는 의미
df['구분'] = df['구분'].ffill()

In [ ]:
# 【3-⑥】 내용이 변경되었는지 확인
# 데이터의 맨 위 n개 행을 화면에 보여달라는 명령
print(df.head(10))

In [ ]:
# 【3-⑦】 엑셀 파일로 저장 (원본에 덮어쓰지 않고 새 파일로)
df.to_excel("전처리완료.xlsx", index=False)

import os
print([f for f in os.listdir() if f.endswith('.xlsx')])   # 새 파일이 생겼는지 확인

In [ ]:
# 【3-⑦】 내 컴퓨터로 내려받기 (Colab 전용)
from google.colab import files
files.download("전처리완료.xlsx")

## 【미니③】 ★ 모두의 관심사 : 우리 팀 점심 데이터 분석 (가상 데이터)
`우리팀_점심기록_가상데이터.csv`를 📁 파일 패널에 올린 뒤, ChatGPT에게 아래처럼 시킵니다. 막히면 아래 셀을 실행하세요.
> Colab에 올린 '우리팀_점심기록_가상데이터.csv'를 읽어줘. 열은 날짜, 요일, 날씨(맑음/비), 직원, 메뉴, 가격, 만족도(1~5), 대기시간(분)이야. 그래프 3개를 나란히 그려줘. ① 요일별 평균 점심값 막대그래프(월~금 순서, 가장 비싼 요일은 빨간색) ② 맑은 날과 비 오는 날의 메뉴별 비율(%) 비교 막대그래프 ③ 메뉴별 평균 가격(x)과 평균 만족도(y) 산점도, 점 옆에 메뉴 이름 표시. 제목과 축 이름은 한글로.

In [ ]:
# 【미니③】 우리 팀 점심 데이터 — 요일 · 날씨 · 가성비
import pandas as pd
import matplotlib.pyplot as plt

점심 = pd.read_csv('우리팀_점심기록_가상데이터.csv')
print(점심.shape)                                              # (400, 8)

요일별 = 점심.groupby('요일')['가격'].mean().reindex(['월', '화', '수', '목', '금'])
날씨별 = pd.crosstab(점심['메뉴'], 점심['날씨'], normalize='columns') * 100   # 날씨별 메뉴 비율(%)
메뉴별 = 점심.groupby('메뉴')[['가격', '만족도']].mean()

fig, ax = plt.subplots(1, 3, figsize=(17, 4.8))
색 = ['#C92A2A' if d == 요일별.idxmax() else '#ADB5BD' for d in 요일별.index]
ax[0].bar(요일별.index, 요일별.values, color=색)
for i, v in enumerate(요일별.values):
    ax[0].text(i, v, f'{v:,.0f}', ha='center', va='bottom')
ax[0].set_title('요일별 평균 점심값'); ax[0].set_xlabel('요일'); ax[0].set_ylabel('가격 (원)')

날씨별[['맑음', '비']].plot(kind='bar', ax=ax[1], color=['#FCC419', '#1971C2'], rot=0)
ax[1].legend(title='날씨', loc='upper center', ncol=2); ax[1].set_ylim(0, 32)
ax[1].set_title('맑은 날 vs 비 오는 날 메뉴 비율'); ax[1].set_xlabel('메뉴'); ax[1].set_ylabel('비율 (%)')

ax[2].scatter(메뉴별['가격'], 메뉴별['만족도'], s=120, color='#E8590C')
for m, r in 메뉴별.iterrows():
    ax[2].text(r['가격'] + 150, r['만족도'], m)
ax[2].set_title('메뉴별 가격 vs 만족도 (가성비)'); ax[2].set_xlabel('평균 가격 (원)'); ax[2].set_ylabel('평균 만족도 (1~5)')
plt.tight_layout()
plt.show()

---
# 4모듈 · 데이터 분석
## 【실습①】 월별 9개 시트를 한 표로 합치기
원본은 머리글이 3줄이고, 날짜가 3월은 날짜형 / 4월 이후는 '4월10일' 글자이며, 날짜는 24행에 한 번만 적혀 있습니다.
- 3줄 건너뛰고(`skiprows=3`) 열 이름을 직접 붙입니다.
- 날짜의 빈칸은 위 값으로 채웁니다(`ffill`).
- '4월10일' 글자는 2019년 날짜로 바꿉니다.
- b검사값(03:05:00)은 규격이 '3.0±1.0분'이므로 **3분 05초 = 185초**로 해석합니다. (가정 — 현장 확인 필요)

In [ ]:
import pandas as pd, numpy as np, re, datetime as dt

FILE2 = '2. 라인생산_ 검사값 대 온도_월별 Raw Data.xlsx'
sheets = pd.read_excel(FILE2, sheet_name=None, header=None, skiprows=3)   # 모든 시트를 한 번에

frames = []
for name, d in sheets.items():
    d.columns = ['날짜','순번','a검사값','b검사값','c검사값','d검사값','1존온도','2존온도','3존온도']
    d['월'] = int(name.split('월')[0])
    d['날짜'] = d['날짜'].ffill()
    frames.append(d)
df = pd.concat(frames, ignore_index=True)

def to_date(v):
    if isinstance(v, (dt.datetime, pd.Timestamp)):
        return pd.Timestamp(v)
    m = re.match(r'(\d+)월\s*(\d+)일', str(v))
    return pd.Timestamp(2019, int(m.group(1)), int(m.group(2)))

df['날짜'] = df['날짜'].map(to_date)
df['b검사값'] = df['b검사값'].map(lambda t: t.hour*60 + t.minute)   # 03:05:00 -> 185초
print(df.shape)
df.head()

## 【실습②】 분석 후 무조건 확인하는 3가지: 건수 · 합계 · 상식

In [ ]:
print('건수:', len(df))
print('d검사값 합계:', df['d검사값'].sum())
print('빈 값:', int(df.isnull().sum().sum()))
print('측정 날짜 수:', df['날짜'].nunique(), '일')

# a검사값 규격 5.8~6.8
out_a = df[(df['a검사값'] < 5.8) | (df['a검사값'] > 6.8)]
print('a검사값 규격 이탈:', len(out_a), '건')

## 【실습③④】 예측하기(회귀)
**질문 A:** 1존 온도로 2존 온도를 맞힐 수 있을까?  **질문 B:** 건조온도(1~3존)로 c검사값을 맞힐 수 있을까?

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

def run_reg(X, y, label):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = LinearRegression().fit(X_train, y_train)
    pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    base = np.sqrt(mean_squared_error(y_test, np.full(len(y_test), y_train.mean())))  # '평균으로 찍기'
    print(f'[{label}]  설명력 R²={r2_score(y_test, pred):.3f}  평균 오차 RMSE={rmse:.2f}  (평균으로 찍었을 때 RMSE={base:.2f})')
    return model

# 질문 A : 1존 -> 2존
model_A = run_reg(df[['1존온도']], df['2존온도'], '1존 → 2존 온도')
print('회귀식: 2존 =', round(model_A.coef_[0], 3), '× 1존 +', round(model_A.intercept_, 1))

# 질문 B : 건조온도 -> c검사값
model_B = run_reg(df[['1존온도','2존온도','3존온도']], df['c검사값'], '건조온도 → c검사값')

In [ ]:
# 눈으로 확인 : 두 질문의 산점도
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].scatter(df['1존온도'], df['2존온도'], s=10, alpha=0.5)
ax[0].set_title('1존 온도 vs 2존 온도'); ax[0].set_xlabel('1존(℃)'); ax[0].set_ylabel('2존(℃)')
ax[1].scatter(df['1존온도'], df['c검사값'], s=10, alpha=0.5, color='tab:orange')
ax[1].set_title('1존 온도 vs c검사값'); ax[1].set_xlabel('1존(℃)'); ax[1].set_ylabel('c검사값(℃)')
plt.tight_layout()
plt.show()

## 【실습⑤】 판정하기(분류) — 라벨 만들기
분류는 '정답 라벨(불량/정상)'이 필요합니다. 먼저 **우리 데이터에서 라벨을 만들 수 있는지** 확인합니다.

In [ ]:
# a검사값이 규격(5.8~6.8)을 벗어나면 1(이탈)
df['규격이탈'] = ((df['a검사값'] < 5.8) | (df['a검사값'] > 6.8)).astype(int)
print(df['규격이탈'].value_counts())
print('이탈 비율:', round(df['규격이탈'].mean()*100, 2), '%')
print('→ 전부 정상이라고 찍어도 정확도 =', round((1-df['규격이탈'].mean())*100, 1), '%')

**해석:** 이탈은 648건 중 5건(0.8%)뿐입니다. 이 데이터로는 판정 모델을 학습할 수 없습니다.
불량이 거의 없는 안정적인 공정에서는 판정 모델보다 **규격 이탈 알림(관리도)**이 맞습니다. 분류 모델 심화는 맨 아래 **부록**에 있습니다(선택).

## 【실습⑥】 묶기(군집) — 숨은 9건을 찾아라
정답 없이 `1존·2존·3존 온도`가 비슷한 측정값끼리 **2개 그룹**으로 묶고, 작은 그룹이 언제인지 찾습니다.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

feat = ['1존온도','2존온도','3존온도']
Z = StandardScaler().fit_transform(df[feat])          # 단위를 맞추는 정리 작업
km = KMeans(n_clusters=2, n_init=10, random_state=42).fit(Z)
df['군집'] = km.labels_

print(df['군집'].value_counts())                      # 그룹별 개수
print(df.groupby('군집')[feat].mean().round(1))        # 그룹별 평균 온도

In [ ]:
# 작은 그룹은 누구인가? -> 현장 확인이 필요한 구간
small = df['군집'].value_counts().idxmin()
df[df['군집'] == small][['날짜','순번'] + feat + ['a검사값','c검사값']]

In [ ]:
import seaborn as sns
sns.scatterplot(data=df, x='1존온도', y='2존온도', hue='군집', palette=['#868E96', '#E8590C'])
plt.title('1존 vs 2존 온도 — 동떨어진 점은 어느 그룹?')
plt.show()

### (심화·선택) 그룹 수를 2~5개로 바꿔 비교하기

In [ ]:
from sklearn.metrics import silhouette_score
for k in range(2, 6):
    km_k = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Z)
    print(f'k={k}  silhouette={silhouette_score(Z, km_k.labels_):.3f}  군집별 개수={sorted(pd.Series(km_k.labels_).value_counts().tolist(), reverse=True)}')

---
# 5모듈 · 데이터 시각화
## 【실습⑦】 그래프 4종 릴레이 — 분포 · 분포 비교 · 비교 · 추세

In [ ]:
# 분포 → 히스토그램 (규격선 표시)
df['a검사값'].plot(kind='hist', bins=15, title='a검사값 분포 (규격 5.8~6.8)')
plt.axvline(5.8, color='red', linestyle='--'); plt.axvline(6.8, color='red', linestyle='--')
plt.xlabel('a검사값')
plt.show()

In [ ]:
# 분포 비교 → 박스플롯 (월별 c검사값)
sns.boxplot(data=df, x='월', y='c검사값')
plt.title('월별 c검사값 분포 (규격 36±4℃)')
plt.show()

In [ ]:
# 비교 → 막대그래프 (월별 d검사값 평균)
df.groupby('월')['d검사값'].mean().plot(kind='bar', title='월별 d검사값 평균 (규격 100±50mm)')
plt.ylabel('mm')
plt.show()

In [ ]:
# 추세 → 선 그래프 (측정일별 1존 온도 평균)
df.groupby('날짜')['1존온도'].mean().plot(marker='o', title='측정일별 1존 평균 온도')
plt.ylabel('℃')
plt.show()

### (선택) 관계 → 산점도 + 상관 히트맵

In [ ]:
sns.heatmap(df[['a검사값','c검사값','d검사값','1존온도','2존온도','3존온도']].corr(), annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('상관관계 히트맵')
plt.show()

## 【실습⑧】 한 줄 프롬프트로 공정 품질 대시보드 만들기
실행하면 `공정_대시보드.html` 파일이 생깁니다. 아래 다운로드 셀로 내려받아 **더블클릭**하면 브라우저에서 열리고, 마우스를 올리면 값이 보이고 드래그하면 확대됩니다.

In [ ]:
# 공정 품질 대시보드 만들기 (HTML 파일 1개 → 내려받아 브라우저로 열기)
import plotly.express as px

kpi = {
    '총 측정': f"{len(df):,}건",
    '측정일': f"{df['날짜'].nunique()}일",
    'a검사값 규격 이탈': f"{((df['a검사값'] < 5.8) | (df['a검사값'] > 6.8)).sum()}건",
    '1존 평균 온도': f"{df['1존온도'].mean():.1f}℃",
}
daily_t = df.groupby('날짜')[['1존온도','2존온도','3존온도']].mean().reset_index()
g1 = px.line(daily_t, x='날짜', y=['1존온도','2존온도','3존온도'], markers=True, title='측정일별 건조온도 (℃)')
monthly = df.groupby('월')['c검사값'].mean().reset_index()
g2 = px.bar(monthly, x='월', y='c검사값', title='월별 c검사값 평균 (규격 36±4℃)', text_auto='.1f')
g3 = px.histogram(df, x='a검사값', nbins=15, title='a검사값 분포 (빨간선 = 규격 5.8~6.8)')
g3.add_vline(x=5.8, line_color='red', line_dash='dash'); g3.add_vline(x=6.8, line_color='red', line_dash='dash')
df['구간'] = np.where(df['1존온도'] >= 252, '고온 구간', '정상 구간')
g4 = px.scatter(df, x='1존온도', y='2존온도', color='구간', title='1존 vs 2존 온도 (고온 구간 찾기)',
                color_discrete_map={'정상 구간':'#868E96', '고온 구간':'#E8590C'}, hover_data=['날짜','순번'])

cards = ''.join(f"<div class='card'><div class='k'>{k}</div><div class='v'>{v}</div></div>" for k, v in kpi.items())
charts = ''.join(f"<div class='box'>{g.to_html(full_html=False, include_plotlyjs=(i == 0))}</div>" for i, g in enumerate([g1, g2, g3, g4]))
html = f"""<html><head><meta charset='utf-8'><title>공정 품질 대시보드</title>
<style>body{{font-family:'Malgun Gothic',sans-serif;background:#F1F3F5;margin:24px}}
h1{{color:#1F2328}} .cards{{display:flex;gap:16px}} .card{{flex:1;background:white;border-radius:12px;padding:16px;border-top:5px solid #E8590C}}
.k{{color:#495057}} .v{{font-size:32px;font-weight:bold;color:#1F2328}}
.grid{{display:grid;grid-template-columns:1fr 1fr;gap:16px;margin-top:16px}} .box{{background:white;border-radius:12px;padding:8px}}</style></head>
<body><h1>라인생산 공정 품질 대시보드 (2019년 3~11월)</h1><div class='cards'>{cards}</div><div class='grid'>{charts}</div></body></html>"""
with open('공정_대시보드.html', 'w', encoding='utf-8') as f:
    f.write(html)
print('공정_대시보드.html 저장 완료 → 왼쪽 📁 파일 패널에서 내려받아 더블클릭')

In [ ]:
# 대시보드 내려받기 (Colab 전용)
from google.colab import files
files.download('공정_대시보드.html')

## 【실습⑧-2】 ★ Colab 꿀팁 : Gemini '시각화 모드'로 말해서 대시보드 받기
**코드를 복사하지 않고, Colab 안에서 말로 주문합니다.** (2026년 6월 Colab에 추가된 기능)

1. 위쪽 【1-4】 데이터 불러오기 셀을 먼저 실행해 `df`가 있는 상태로 만듭니다.
2. Colab 화면의 **Gemini** 버튼 옆 드롭다운(▼) → **시각화 모드** 선택
3. 아래 문장을 붙여넣고 전송 → Gemini가 코드를 만들고, 실행하고, 오류가 나면 스스로 고칩니다.

> df로 라인생산 공정 품질 대시보드를 만들어줘. 위에 숫자 카드 4개(총 측정 건수, 측정일 수, a검사값 규격 5.8~6.8 이탈 건수, 1존 평균 온도), 아래에 측정일별 1·2·3존 온도 선 그래프와 1존 vs 2존 산점도. 제목과 축 이름은 한글로.

4. 결과 셀의 출력 메뉴(⋮) → **전체 화면 출력 링크 복사**(영문 메뉴: Copy link to fullscreen output) 또는 셀 아래 **대시보드 공유** 추천 버튼 → 새 탭에 붙여넣으면 코드 없이 대시보드만 보입니다.

✔ 검증 : 카드 숫자가 **648건 · 27일 · 5건 · 245.0℃** 와 같은지 확인
⚠ 이 기능은 데이터를 Gemini(구글 AI)로 보냅니다. 업체 데이터 사용 허용을 먼저 확인하세요. 메뉴가 안 보이면 강사 시연으로 대체합니다.

## 【실습⑨】 (선택) MES 태그 데이터 시계열 — 5번 파일
세로(long)로 쌓인 태그 데이터를 **시간 × 관(#1~#5)** 표로 바꿔(피벗) 그립니다.

In [ ]:
FILE5 = '5. 공정데이터_MES_Tag Data.xlsx'
mes = pd.read_excel(FILE5, sheet_name='관 온도')
mes['EVENTTIME'] = pd.to_datetime(mes['EVENTTIME'])

wide = mes.pivot_table(index='EVENTTIME', columns='RECIPEPARAMETERNAME', values='RESULTVALUE')
wide.columns = [c.replace('1호기_', '').replace('(측정값)', '') for c in wide.columns]
print(wide.shape)
wide.plot(figsize=(10, 4), title='1호기 관 온도 추이 (5분 간격)')
plt.ylabel('℃')
plt.show()
wide.describe().round(1)

---
# 6모듈 · 프로젝트
## 【실습⑫ 예시】 4월 10일 고온 구간의 검사값은 규격 안이었나?
프로젝트 질문을 정하지 못한 경우 이 예시를 그대로 실행해 봅니다.

In [ ]:
# 1) 고온 구간(1존 252℃ 이상)만 골라 검사값이 규격 안인지 판정
hot = df[df['1존온도'] >= 252][['날짜','순번','1존온도','a검사값','c검사값','d검사값']].copy()
hot['a 규격(5.8~6.8)'] = np.where(hot['a검사값'].between(5.8, 6.8), '안', '밖')
hot['c 규격(32~40)']  = np.where(hot['c검사값'].between(32, 40), '안', '밖')
hot['d 규격(50~150)'] = np.where(hot['d검사값'].between(50, 150), '안', '밖')
print('고온 구간:', len(hot), '건')
display(hot)

# 2) 그래프 2개 : 측정일별 1존 온도 / 고온 구간의 a검사값과 규격선
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
t = df.groupby('날짜')['1존온도'].mean()
ax[0].plot(t.index, t.values, marker='o', color='#868E96')
ax[0].scatter([pd.Timestamp('2019-04-10')], [t.loc['2019-04-10']], s=120, color='#E8590C', zorder=3)
ax[0].set_title('측정일별 1존 평균 온도 (주황 = 4월 10일)'); ax[0].set_ylabel('℃')
ax[1].bar(hot['순번'].astype(str), hot['a검사값'], color='#E8590C')
ax[1].axhline(5.8, color='red', linestyle='--'); ax[1].axhline(6.8, color='red', linestyle='--')
ax[1].set_ylim(5.5, 7.0); ax[1].set_title('4월 10일 고온 구간 a검사값 (빨간선 = 규격)'); ax[1].set_xlabel('순번')
plt.tight_layout()
plt.show()

## 【실습⑫ B·C】 내 업무 데이터 · 공공데이터로 하기 (선택)
- **B. 내 업무 데이터** : 회사 정책을 먼저 확인하고, 이름·연락처·고객사명 같은 열은 지우거나 가명으로 바꾼 뒤 올립니다.
- **C. 공공데이터** : 공공데이터포털(data.go.kr)에서 CSV를 내려받아 올립니다.

왼쪽 📁 파일 패널에 파일을 올리고, 아래 `MY_FILE`의 이름만 바꿔 실행하세요.

In [ ]:
# 【실습⑫ B·C】 내 파일 불러오기 — 파일 이름만 바꿔서 실행
MY_FILE = '내파일.csv'   # 📁 파일 패널에 올린 파일 이름으로 바꾸기 (.csv 또는 .xlsx)

if MY_FILE.endswith('.csv'):
    try:
        my = pd.read_csv(MY_FILE)
    except UnicodeDecodeError:
        my = pd.read_csv(MY_FILE, encoding='cp949')   # 공공데이터 CSV는 한글 인코딩(cp949)인 경우가 많음
else:
    my = pd.read_excel(MY_FILE)

print(my.shape)        # 몇 행 × 몇 열?
my.head()              # 앞 5줄 — 열 이름을 확인한 뒤 ChatGPT 프롬프트에 적어 주세요

## 【실습⑩】 결과 저장 — 보고서에 넣을 표와 그래프

In [ ]:
# 표 저장 (엑셀)
summary = df.groupby('월').agg(
    a검사값평균=('a검사값','mean'), c검사값평균=('c검사값','mean'), d검사값평균=('d검사값','mean'),
    온도1존평균=('1존온도','mean'), 온도2존평균=('2존온도','mean'), 온도3존평균=('3존온도','mean')).round(2)
summary.to_excel('월별_검사값_요약.xlsx')

# 그래프 저장 (보고서/PPT용 고해상도)
ax = df.groupby('월')['c검사값'].mean().plot(kind='bar', title='월별 c검사값 평균')
plt.savefig('월별_c검사값.png', dpi=200, bbox_inches='tight')
plt.show()
summary

In [ ]:
# 내 컴퓨터로 내려받기 (Colab 전용) — Colab 파일은 세션이 끝나면 사라집니다
from google.colab import files
for f in ['월별_검사값_요약.xlsx', '월별_c검사값.png']:
    files.download(f)

---
# 부록 (선택) — 분류 모델 심화 : 연습용 가상 데이터
불량률이 충분한 가상 데이터(`라인_센서로그_가상데이터.csv`, 불량률 약 8.5%)로 혼동행렬 · Recall · 판정 기준을 연습합니다.

In [ ]:
sim = pd.read_csv('라인_센서로그_가상데이터.csv')
sim['일시'] = pd.to_datetime(sim['일시'])
sim['시간대'] = sim['일시'].dt.hour
sim['온도'] = sim.groupby('설비 ID')['온도'].transform(lambda s: s.fillna(s.median()))   # 빈 값은 설비별 중앙값
print(sim.shape)
print(sim['불량여부'].value_counts())

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

Xc = pd.get_dummies(sim[['온도','압력','시간대','설비 ID']], columns=['설비 ID'], drop_first=True)
yc = sim['불량여부']
Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.2, random_state=42, stratify=yc)

scaler = StandardScaler().fit(Xc_train)          # 스케일러는 Train에만 fit
Xc_train_s, Xc_test_s = scaler.transform(Xc_train), scaler.transform(Xc_test)

lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42).fit(Xc_train_s, yc_train)
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42).fit(Xc_train, yc_train)

for name, mdl, Xt in [('로지스틱 회귀', lr, Xc_test_s), ('랜덤포레스트', rf, Xc_test)]:
    p = mdl.predict(Xt); proba = mdl.predict_proba(Xt)[:, 1]
    print('=====', name, '=====')
    print(classification_report(yc_test, p, target_names=['정상','불량']))
    print('ROC-AUC:', round(roc_auc_score(yc_test, proba), 3)); print()

In [ ]:
cm = confusion_matrix(yc_test, rf.predict(Xc_test))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['정상(예측)','불량(예측)'], yticklabels=['정상(실제)','불량(실제)'])
plt.title('혼동행렬 - 랜덤포레스트')
plt.show()
print('TN, FP, FN, TP =', cm.ravel())